In [ ]:
%%capture
# see comments in README on changes to the conda venv
import os
# import modin.pandas as pd
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

This notebook calculates the core numbers for the consort chart up to the
total number who completed the baseline visit

n = 1858

Also shows the condition counts declared at screening. These are different
from the condition count confirmed at baseline. The baseline condition count
can be found in the generated table / df "diagnoses".

In [ ]:
from edc_constants.constants import YES
from intecomm_analytics.dataframes import get_patientlog_df


In [ ]:
df = get_patientlog_df()

In [ ]:
len(df)

In [ ]:
df[(df.group_identifier.notna()) & (df.hiv==1) & (df.screening_identifier.notna()) & (df.subject_identifier.notna())].count()

In [ ]:
df.groupby(['hiv',"htn", "dm"]).size().to_frame()


In [ ]:
# total referred / available to screen
df.count()

In [ ]:
from edc_constants.constants import NO

# total willing to screen
df[df.willing_to_screen == NO].count()


In [ ]:
df[(df.stable == NO) & (df.willing_to_screen==YES)].count()


In [ ]:
# total screened
df[df.screening_identifier.eligible==Troe].count()


In [ ]:
from django_pandas.io import read_frame
# total eligible
# df["eligible"] = df["subject_identifier"].apply(lambda x: 1 if x.startswith("107-") else 0)
# df[df.eligible==1].count()

from intecomm_screening.models import SubjectScreening

# total eligible
df_screen = read_frame(SubjectScreening.objects.all())
df_screen[df_screen.eligible == 1].count()




In [ ]:
# total consented
df[~df.consent_datetime.isna()].count()


In [ ]:
# grouped
df[~df.group_identifier.isna()].count()


In [ ]:
from intecomm_rando.models import RandomizationList
# conditions at grouping
from edc_pdutils.dataframes import get_subject_visit
import pandas as pd
df_visit = get_subject_visit("intecomm_subject.subjectvisit")
df_visit = df_visit[(df_visit.visit_code == 1000.0) & ~(df_visit.subject_identifier=="107-208-0014-2")]
# df_visit
df_main = pd.merge(df_visit[["subject_identifier"]], df[(df.group_identifier.notna())], on="subject_identifier", how="left")
# 1858 subjects

def get_ncd(s):
    if (s["htn"] == 1 or s["dm"] == 1) and s["hiv"] == 0:
        return 1
    return 0

def get_hiv_only(s):
    if s["htn"] == 0 and s["dm"] == 0 and s["hiv"] == 1:
        return 1
    return 0

df_main["ncd"] = df_main.apply(get_ncd, axis=1)
df_main["hiv_only"] = df_main.apply(get_hiv_only, axis=1)

df_rando = read_frame(RandomizationList.objects.values("group_identifier", "assignment").filter(group_identifier__isnull=False))
df_main = df_main.merge(df_rando[["group_identifier", "assignment"]], on="group_identifier", how="left")
# df_main.to_csv(Path("/Users/erikvw/Documents/ucl/protocols/intecomm/analysis/primary/") / "df_main_1858.csv", index=False)



In [ ]:
df_main[df_main.hiv==1]

In [ ]:
df_main.to_csv(report_folder)

In [ ]:
df_conds = df_main.groupby(['hiv',"htn", "dm"]).size().to_frame()
df_conds = df_conds.reset_index()
df_conds.columns= ["hiv", "htn", "dm", "count"]
df_conds


In [ ]:
df_conds[(df_conds.hiv==1) & (df_conds.htn==0) & (df_conds.dm==0) ]["count"].sum()

In [ ]:
df_conds[(df_conds.hiv==0) & ((df_conds.htn==1) | (df_conds.dm==1)) ]["count"].sum()

In [ ]:
df_conds[(df_conds.hiv==0) & (df_conds.htn==1) & (df_conds.dm==0) ]["count"].sum()

In [ ]:
df_conds[(df_conds.hiv==0) & (df_conds.htn==0) & (df_conds.dm==1) ]["count"].sum()


In [ ]:
df_conds[(df_conds.hiv==1) & ((df_conds.htn==1) | (df_conds.dm==1)) ]["count"].sum()


In [ ]:
df_conds["count"].sum()

In [ ]:
# completed the baseline visit, almost! see below

from edc_pdutils.dataframes import get_subject_visit
df_visit = get_subject_visit("intecomm_subject.subjectvisit")
df_visit[df_visit.visit_code == 1000.0].count()


In [ ]:
# seems to have attended baseline but did not
# 107-208-0014-2 should be excluded as well because CRFs
# were not submitted at baseline

from intecomm_subject.models import SubjectVisit
from edc_crf.utils import HasCrfChecker

c = HasCrfChecker("intecomm_subject", SubjectVisit, )
for _, row in df_visit[df_visit.visit_code == 1000.0].iterrows():
    _, has_crfs = c.has_crfs(row["subject_identifier"], row["visit_code_str"], int(row["visit_code_sequence"]))
    if not has_crfs:
        print(row["subject_identifier"], row["visit_code_str"], int(row["visit_code_sequence"]))

In [ ]:
c = HasCrfChecker("intecomm_subject", SubjectVisit, )
c.has_crfs("107-208-0014-2", "1000", 0)

In [ ]:
df_visit[(df_visit.visit_code == 1000.0) & (df_visit.subject_identifier != "107-208-0014-2")].count()
